# 1. 데이터 분석을 위한 두 번째 걸음~~~

## a. 다버리긴 아깝지... drop / dropna 말고 fillna 😢
---
- 어떻게 사용하나요? df.fillna(속성값) = 속성값을 전달하여 옵션 설정
    - 자주 사용하는 속성값
        - value : 결측값을 대체할 값 --> 벙벙하게 전체 데이터프레임을 채우면 경고
        - method : 결측값을 변경할 방식 = ffill(전 값으로 채우기), bfill(뒷 값으로 채우기) --> 경고 deprecated 예정 = method 
        - axis : {0 : index / 1 : columns}
        - inplace : 원본을 변경
- 그냥 사용해 보자 --> 우선 샘플 데이터 생성

In [2]:
import numpy as np
import pandas as pd
df = pd.DataFrame(np.round(np.random.rand(4, 4)*100), index=['a', 'b', 'c', 'd'], columns = ['math', 'eng', 'sci', 'kor']) # pd.DataFrame 속성을 이용하는 방법
df['dvi'] = [pd.NA, 30, 40, 90]
df

,math,eng,sci,kor,dvi
a,28.0,65.0,1.0,81.0,<NA>
b,62.0,39.0,56.0,99.0,30
c,74.0,40.0,73.0,12.0,40
d,78.0,91.0,53.0,68.0,90


- 배운 것 사용하면서 익숙해 지자

In [3]:
df.iloc[1:3, 1] = [np.nan, np.nan] # iloc 오랜만에... 아시죠? iloc은 인덱스 슬라이딩만 가능! = (2행~3행, 1열) np.nan으로 교체
df

,math,eng,sci,kor,dvi
a,6.0,43.0,87.0,52.0,<NA>
b,62.0,NaN,90.0,23.0,30
c,2.0,NaN,20.0,97.0,40
d,32.0,1.0,80.0,42.0,90


In [4]:
df.loc['d', 'sci'] = pd.NA # but loc은 다 가능 --> 이걸 그냥 쓰세요. ('d'행, 'sci'열) pd.NA로 교체
df

,math,eng,sci,kor,dvi
a,6.0,43.0,87.0,52.0,<NA>
b,62.0,NaN,90.0,23.0,30
c,2.0,NaN,20.0,97.0,40
d,32.0,1.0,NaN,42.0,90


- 여기서 다 dropna하기에는... 너무 많은 데이터 손실

In [5]:
df.dropna(axis=0) # 행 드롭

,math,eng,sci,kor,dvi


In [6]:
df.dropna(axis=1) # 열 드롭

,math,kor
a,6.0,52.0
b,62.0,23.0
c,2.0,97.0
d,32.0,42.0


- 이런 경우 결측치 채우기... 다시 복습... 컬럼이 50% 이상의 결측치가 있을땐 삭제

In [7]:
df.isnull().sum(axis=0) / len(df) >= 0.5 # 주의 컬럼 방향~

math    False
eng      True
sci     False
kor     False
dvi     False
dtype: bool

In [8]:
df2 = df.drop(df.columns[df.isnull().sum(axis=0) / len(df) >= 0.5], axis=1) # df2에 저장
df2

,math,sci,kor,dvi
a,6.0,87.0,52.0,<NA>
b,62.0,90.0,23.0,30
c,2.0,20.0,97.0,40
d,32.0,NaN,42.0,90


- 특정값으로 결측치 채우기 = df.fillna(value) 

In [9]:
df2.fillna(0) # 전체 데이터셋의 결측치를 0으로 체우기 --> 경고

C:\Users\tysep16\AppData\Local\Temp\ipykernel_30060\2434569580.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df2.fillna(0) # 전체 데이터셋의 결측치를 0으로 체우기 --> 경고


,math,sci,kor,dvi
a,6.0,87.0,52.0,0
b,62.0,90.0,23.0,30
c,2.0,20.0,97.0,40
d,32.0,0.0,42.0,90


- 그래서 대처법은?...

In [10]:
df2.sci.fillna(0) # 최소한 컬럼단위로 구체화

a    87.0
b    90.0
c    20.0
d     0.0
Name: sci, dtype: float64

- 왜냐면... 이렇게 많이 쓰이기 때문...

In [11]:
df2.sci.fillna(df2.sci.mean()) # 해당 컬럼의 평균으로 채우기

a    87.000000
b    90.000000
c    20.000000
d    65.666667
Name: sci, dtype: float64

- 한번에 여러개도 가능 = 딕셔너리로 처리 

In [12]:
df2.fillna({'sci':0, 'dvi':30})

C:\Users\tysep16\AppData\Local\Temp\ipykernel_30060\3490838635.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df2.fillna({'sci':0, 'dvi':30})


,math,sci,kor,dvi
a,6.0,87.0,52.0,30
b,62.0,90.0,23.0,30
c,2.0,20.0,97.0,40
d,32.0,0.0,42.0,90


- 혹은 결측치 이전 값 결측치 이후 값도 가져올 수 있음

In [13]:
df2.fillna(method='ffill') # 이전

C:\Users\tysep16\AppData\Local\Temp\ipykernel_30060\1410440438.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df2.fillna(method='ffill') # 이전


,math,sci,kor,dvi
a,6.0,87.0,52.0,<NA>
b,62.0,90.0,23.0,30
c,2.0,20.0,97.0,40
d,32.0,20.0,42.0,90


In [14]:
df2.fillna(method='bfill') # 이후

C:\Users\tysep16\AppData\Local\Temp\ipykernel_30060\2360952841.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df2.fillna(method='bfill') # 이후
C:\Users\tysep16\AppData\Local\Temp\ipykernel_30060\2360952841.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df2.fillna(method='bfill') # 이후


,math,sci,kor,dvi
a,6.0,87.0,52.0,30
b,62.0,90.0,23.0,30
c,2.0,20.0,97.0,40
d,32.0,NaN,42.0,90


- 메쏘드로 바뀐다네요.

In [15]:
df2.ffill() # 근데 이건 에러가 없는데...

,math,sci,kor,dvi
a,6.0,87.0,52.0,<NA>
b,62.0,90.0,23.0,30
c,2.0,20.0,97.0,40
d,32.0,20.0,42.0,90


In [16]:
df2.bfill() # 이건 있네요? 아직, 왔다갔다 하는 듯.. 

C:\Users\tysep16\AppData\Local\Temp\ipykernel_30060\1563728673.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df2.bfill() # 이건 있네요? 아직, 왔다갔다 하는 듯..


,math,sci,kor,dvi
a,6.0,87.0,52.0,30
b,62.0,90.0,23.0,30
c,2.0,20.0,97.0,40
d,32.0,NaN,42.0,90


## b. `groupby.집계함수`? groupby.agg('집계함수')? = 2가지 방법 🥱
---
- groupby = 특정 컬럼을 조건에 따라 묶은 후 --> 그룹 별로 집계연산(sum, count, mean, max, min...) 함수
    - 예: 일별 매출 평균(mean), 월별 인당 야근 수(count), 년도별 학과 최대 합격률(max)..<br><br>

- ✅ 자주쓰이는 집계함수
| 함수       | 설명                        |
|------------|-----------------------------|
| `sum()`    | 합계                        |
| `mean()`   | 평균                        |
| `median()` | 중앙값                      |
| `min()`    | 최소값                      |
| `max()`    | 최대값                      |
| `count()`  | 누락값(NaN) 제외한 개수     |
| `size()`   | NaN 포함 전체 개수 (groupby 전용) |
| `std()`    | 표준편차                    |
| `var()`    | 분산                        |


- **방법#1**: df.groupby(by=그룹이 되는 컬럼)[계산하고 싶은 컬럼].집계함수()
- 일단 해보는게 --> 타이타닉 데이터셋(csv파일)을 불러와서 데이터프레임 df를 만들어 보세요.

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/tysep16/DV25_1/refs/heads/main/titanic.csv')
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


- DataFrame 집계함수

In [19]:
df.count() # count 예시 = 옵져베이션 숫자 = 행의 숫자

PassengerId    891
Survived       891
Pclass         891
Name           891
Sex            891
Age            714
SibSp          891
Parch          891
Ticket         891
Fare           891
Cabin          204
Embarked       889
dtype: int64

- 특정 칼럼을 집계함수 

In [21]:
df[['Age', 'Fare']].mean() # Age 컬럼과 Fare 컬럼의 평균

Age     29.699118
Fare    32.204208
dtype: float64

- groupby를 적용해 보자. = `DataFrame의 groupby()는 DataFrame을 반환`
- 특정 칼럼을 '기준으로' 구분 후 다른 특정 칼럼을 집계함수 

In [22]:
df.groupby(by='Pclass').count() # 특정 컬럼(=Pclass)을 기준으로 구분 후 --> 그냥 집계함수(=count)

,PassengerId,Survived,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
Pclass,,,,,,,,,,,
1,216,216,216,216,186,216,216,216,216,176,214
2,184,184,184,184,173,184,184,184,184,16,184
3,491,491,491,491,355,491,491,491,491,12,491


- 그래서 우리가 관심있는 컬럼만 groupby+집계함수를 적용하려면? = 데이터프레임 --> 원하는 컬럼만 불러오면 ok

In [23]:
df.groupby(by='Pclass')[['Survived', 'Fare']].count() # 특정 컬럼(=Pclass)을 기준으로 구분 후 --> 다른 특정 칼럼(=Survived, Fare)을 집계함수(=count)

,Survived,Fare
Pclass,,
1,216,216
2,184,184
3,491,491


- 라면 끓이는 방법에 대해 자유로워져야 합니다... --> 복습

In [24]:
df[df.Pclass == 1]['Survived'].count() # df직접쓰기 + 불리언 인덱싱, 카운트 =  절대적인 행 수

np.int64(216)

In [25]:
df[df.Pclass == 1]['Survived'].sum() # df직접쓰기 + 불리언 인덱싱, 살아남은 사람 수 = Survived --> 1 이므로

np.int64(136)

In [26]:
df.loc[df['Pclass'] == 1, 'Survived'].sum() # loc를 이용해서 지정도 가능

np.int64(136)

In [27]:
df.query('Pclass == 1')['Survived'].sum() # query를 이용해서 지정도 가능

np.int64(136)

In [28]:
np.sum([a for a in df.Pclass if a == 1]) # 컴프리헨션을 이용한 방법

np.int64(216)

In [29]:
np.sum([a if a == 1 else 0 for a in df.Pclass]) # 컴프리헨션을 이용한 방법 + if else

np.int64(216)

In [30]:
pd.Series([a for a in df.Pclass if a == 1]).sum() # 컴프리헨션을 이용한 방법

np.int64(216)

In [31]:
df.Pclass.apply(lambda x: x if x == 1 else 0).sum() # 람다로 가능 = 람다는 무조건 else 포함

np.int64(216)

## b. groupby.집계함수? `groupby.agg('집계함수')`? 🥱
---
- groupby = 특정 컬럼을 조건에 따라 묶은 후 --> 그룹 별로 agg('집계연산')(sum, count, mean, max, min...) 함수
- **방법#2**: df.groupby(그룹이 되는 컬럼)[계산하고 싶은 컬럼].agg('집계함수')

- DataFrame 집계함수

In [32]:
df.agg('count') # count 예시

PassengerId    891
Survived       891
Pclass         891
Name           891
Sex            891
Age            714
SibSp          891
Parch          891
Ticket         891
Fare           891
Cabin          204
Embarked       889
dtype: int64

- 특정 칼럼을 집계함수 

In [33]:
df[['Age', 'Fare']].agg('mean') # mean 예시

Age     29.699118
Fare    32.204208
dtype: float64

- groupby를 적용해 보자. = `DataFrame의 groupby()는 DataFrame을 반환`
- agg는 특정 칼럼을 '기준으로' 구분 후 다른 특정 칼럼을 집계함수 --> agg도 DataFrame 반환

In [4]:
df.groupby(by='Pclass').agg('count')

,PassengerId,Survived,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
Pclass,,,,,,,,,,,
1,216,216,216,216,186,216,216,216,216,176,214
2,184,184,184,184,173,184,184,184,184,16,184
3,491,491,491,491,355,491,491,491,491,12,491


- 그래서 우리가 관심있는 컬럼만 groupby+집계함수를 적용하려면? = 데이터프레임 --> 원하는 컬럼 + 데이터타입을 맞춰야...

In [36]:
df.groupby(by='Pclass')[['Survived', 'Fare']].agg('mean') # 특정 컬럼을 기준으로 구분 후 --> dtype 맞춰서 --> 다른 특정 칼럼을 집계함수

,Survived,Fare
Pclass,,
1,0.629630,84.154687
2,0.472826,20.662183
3,0.242363,13.675550


- <span style='background:yellow; color:red'>**뭐야! 똑같잖아! 왜 aggregation을 쓰는 거지?**</span>

- 만약 Pclass로 그룹핑한후 'Survived'와 'Fare' 컬럼의 sum과 mean을 보려면? --> groupby.집계함수

In [37]:
df.groupby(by='Pclass')[['Survived', 'Fare']].sum(), df.groupby(by='Pclass')[['Survived', 'Fare']].mean()

(        Survived        Fare
 Pclass                      
 1            136  18177.4125
 2             87   3801.8417
 3            119   6714.6951,
         Survived       Fare
 Pclass                     
 1       0.629630  84.154687
 2       0.472826  20.662183
 3       0.242363  13.675550)

- 하지만, agg는...

In [38]:
df.groupby(by='Pclass')[['Survived', 'Fare']].agg(['sum', 'mean'])

Survived                  Fare           
            sum      mean         sum       mean
Pclass                                          
1           136  0.629630  18177.4125  84.154687
2            87  0.472826   3801.8417  20.662183
3           119  0.242363   6714.6951  13.675550

- **문제는 컬럼명이 멀티인덱스(multi-index, 다단계인덱스)가 된다는 점!!** --> 그래서 보통 다음 2가지 방법으로 agg 사용

1. 원래 컬럼에 덮어써서 멀티인덱스 컬럼이 안생기는 방법 = 딕셔너리를 사용

In [39]:
df.groupby(by='Pclass')[['Survived', 'Fare']].agg({'Survived': 'sum', 'Fare': 'mean'})

,Survived,Fare
Pclass,,
1,136,84.154687
2,87,20.662183
3,119,13.675550


2. 새컬럼을 생성하여 멀티인덱스 컬럼이 안생기는 방법 = 프로퍼티를 튜플로 전달 = .agg(새컬럼=(원래컬럼, 집계함수), ...) --> 추천

In [40]:
df.groupby(by='Pclass')[['Survived', 'Fare']].agg(Suvived_sum = ('Survived', 'sum'), Fare_mean = ('Fare', 'mean'))

,Suvived_sum,Fare_mean
Pclass,,
1,136,84.154687
2,87,20.662183
3,119,13.675550


- 그래서 요약하자면, <br><br>

| agg 형태                          | 새 컬럼 이름 생기나? | 결과                          |
|----------------------------------|------------------------|-----------------------------|
| `agg('sum')`                     | ❌                     | 시리즈                      |
| `agg(['sum', 'mean'])`           | ❌ --> 멀티인덱스       | 데이터프레임                |
| `agg({'col': 'sum'})`            | ❌ --> 기존 열이름      | 데이터프레임                |
| `agg(new_name=('col', 'sum'))`   | ✅ --> 새로운 열이름    | 데이터프레임                |

#######

# 2. 여러분! 죄송합니다. 쉬운 방법 --> 어려운 방법 = 쉬운 방법만...

## a. 컬럼간 숫자 계산은 브로드케스팅 비슷하게 작동 --> 그럼 문자는? 🛍️
---
- 예시 데이터 확인 --> 이게 무슨 말이냐 하면

In [41]:
df.Fare

0       7.2500
1      71.2833
2       7.9250
3      53.1000
4       8.0500
        ...   
886    13.0000
887    30.0000
888    23.4500
889    30.0000
890     7.7500
Name: Fare, Length: 891, dtype: float64

In [42]:
df.Fare + 10 # 컬럼의 모든 인자에 10을 더하기 = 사칙연산 가능

0      17.2500
1      81.2833
2      17.9250
3      63.1000
4      18.0500
        ...   
886    23.0000
887    40.0000
888    33.4500
889    40.0000
890    17.7500
Name: Fare, Length: 891, dtype: float64

- 지난 과제 중에...
- Fare 열에 1450원을 곱해서 현재 원화환률로 환산하는 코드는?

In [43]:
df.Fare.apply(lambda x: x * 1450) # 답은... apply & 람다를 이용해서

0       10512.500
1      103360.785
2       11491.250
3       76995.000
4       11672.500
          ...    
886     18850.000
887     43500.000
888     34002.500
889     43500.000
890     11237.500
Name: Fare, Length: 891, dtype: float64

In [47]:
df.Fare * 1450 # 하지만 쉬운 방법

0       10512.500
1      103360.785
2       11491.250
3       76995.000
4       11672.500
          ...    
886     18850.000
887     43500.000
888     34002.500
889     43500.000
890     11237.500
Name: Fare, Length: 891, dtype: float64

- 그렇다면, 문자도 브로드케스팅이 될까? 역시 지난 과제 중에...
- Name 열에서 성(예: Braund, Mr. Owen Harris의 성은 Braund)만 출력하세요.

In [48]:
df.Name

0                                Braund, Mr. Owen Harris
1      Cumings, Mrs. John Bradley (Florence Briggs Th...
2                                 Heikkinen, Miss. Laina
3           Futrelle, Mrs. Jacques Heath (Lily May Peel)
4                               Allen, Mr. William Henry
                             ...                        
886                                Montvila, Rev. Juozas
887                         Graham, Miss. Margaret Edith
888             Johnston, Miss. Catherine Helen "Carrie"
889                                Behr, Mr. Karl Howell
890                                  Dooley, Mr. Patrick
Name: Name, Length: 891, dtype: object

In [49]:
df.assign(Name = [x.split(',')[0] for x in df.Name]).Name # 답은...

0         Braund
1        Cumings
2      Heikkinen
3       Futrelle
4          Allen
         ...    
886     Montvila
887       Graham
888     Johnston
889         Behr
890       Dooley
Name: Name, Length: 891, dtype: object

In [50]:
df.Name.split(',')[0] # 이렇게 되겠지?

AttributeError: 'Series' object has no attribute 'split'

- 응, 안돼. 역시 안되는 건가요?

## b. s.str = 당연 됩니다. 🤣
---
- 시리즈에 '.str'를 붙이면 숫자처럼 오브젝트화하여 브로드케스팅 like하게 적용
- 지난 과제 문제를 다시 풀어보자

In [51]:
df.Name.str.split(',')[0] # 엇 이상한 결과가?

['Braund', ' Mr. Owen Harris']

- 이건 오브젝트화 하여 수행하는 메쏘드(펑션)은 한 개씩.. 즉,

In [52]:
df.Name.str.split(',') # 에서 첫 번째 로우를 출력한 것. split 펑션 따로 인덱싱 따로...

0                             [Braund,  Mr. Owen Harris]
1      [Cumings,  Mrs. John Bradley (Florence Briggs ...
2                              [Heikkinen,  Miss. Laina]
3        [Futrelle,  Mrs. Jacques Heath (Lily May Peel)]
4                            [Allen,  Mr. William Henry]
                             ...                        
886                             [Montvila,  Rev. Juozas]
887                      [Graham,  Miss. Margaret Edith]
888          [Johnston,  Miss. Catherine Helen "Carrie"]
889                             [Behr,  Mr. Karl Howell]
890                               [Dooley,  Mr. Patrick]
Name: Name, Length: 891, dtype: object

In [53]:
df.Name.str.split(',').str[0]

0         Braund
1        Cumings
2      Heikkinen
3       Futrelle
4          Allen
         ...    
886     Montvila
887       Graham
888     Johnston
889         Behr
890       Dooley
Name: Name, Length: 891, dtype: object

- FIFA 2025 테이터셋 문제로 다시 돌아가보자. 키를 cm 부분만 잘라오는 문제

In [54]:
df = pd.read_csv('https://raw.githubusercontent.com/tysep16/DV25_1/refs/heads/main/fifa25.csv')
df.Height

0         182cm / 6'0"
1         191cm / 6'3"
2         195cm / 6'5"
3         186cm / 6'1"
4         176cm / 5'9"
             ...      
16156     169cm / 5'7"
16157     176cm / 5'9"
16158    181cm / 5'11"
16159     187cm / 6'2"
16160     182cm / 6'0"
Name: Height, Length: 16161, dtype: object

- 복습 = 맵 + 람다 사용하기

In [55]:
df.assign(Height = list(map(lambda x: int(x.split('cm')[0]), df.Height))).Height

0        182
1        191
2        195
3        186
4        176
        ... 
16156    169
16157    176
16158    181
16159    187
16160    182
Name: Height, Length: 16161, dtype: int64

- str 사용하기

In [56]:
df.Height.str.split('cm').str[0].apply(int)

0        182
1        191
2        195
3        186
4        176
        ... 
16156    169
16157    176
16158    181
16159    187
16160    182
Name: Height, Length: 16161, dtype: int64

- 네, 인덱스도 컬럼도 다 가능해요.
- 띄어쓰기를 언더바로 바꾸는 문제도 다시 풀어볼까요?

In [57]:
df.columns # 컬럼 라벨 = 띄어쓰기 있습니다.

Index(['Unnamed: 0.1', 'Unnamed: 0', 'Rank', 'Name', 'OVR', 'PAC', 'SHO',
       'PAS', 'DRI', 'DEF', 'PHY', 'Acceleration', 'Sprint Speed',
       'Positioning', 'Finishing', 'Shot Power', 'Long Shots', 'Volleys',
       'Penalties', 'Vision', 'Crossing', 'Free Kick Accuracy',
       'Short Passing', 'Long Passing', 'Curve', 'Dribbling', 'Agility',
       'Balance', 'Reactions', 'Ball Control', 'Composure', 'Interceptions',
       'Heading Accuracy', 'Def Awareness', 'Standing Tackle',
       'Sliding Tackle', 'Jumping', 'Stamina', 'Strength', 'Aggression',
       'Position', 'Weak foot', 'Skill moves', 'Preferred foot', 'Height',
       'Weight', 'Alternative positions', 'Age', 'Nation', 'League', 'Team',
       'play style', 'url', 'GK Diving', 'GK Handling', 'GK Kicking',
       'GK Positioning', 'GK Reflexes'],
      dtype='object')

In [58]:
df.columns.str.replace(' ', '_') # 자! 디제 없습니다.

Index(['Unnamed:_0.1', 'Unnamed:_0', 'Rank', 'Name', 'OVR', 'PAC', 'SHO',
       'PAS', 'DRI', 'DEF', 'PHY', 'Acceleration', 'Sprint_Speed',
       'Positioning', 'Finishing', 'Shot_Power', 'Long_Shots', 'Volleys',
       'Penalties', 'Vision', 'Crossing', 'Free_Kick_Accuracy',
       'Short_Passing', 'Long_Passing', 'Curve', 'Dribbling', 'Agility',
       'Balance', 'Reactions', 'Ball_Control', 'Composure', 'Interceptions',
       'Heading_Accuracy', 'Def_Awareness', 'Standing_Tackle',
       'Sliding_Tackle', 'Jumping', 'Stamina', 'Strength', 'Aggression',
       'Position', 'Weak_foot', 'Skill_moves', 'Preferred_foot', 'Height',
       'Weight', 'Alternative_positions', 'Age', 'Nation', 'League', 'Team',
       'play_style', 'url', 'GK_Diving', 'GK_Handling', 'GK_Kicking',
       'GK_Positioning', 'GK_Reflexes'],
      dtype='object')

<img src="https://raw.githubusercontent.com/tysep16/DV25_1/refs/heads/main/easy.jpg" width="600"><br>

- 하지만, 리스트컴프리헨션, 리스트맵람다, 어플라이람다 = 매우 중요합니다. 모든 복잡한 데이터처리는 이것으로 다 가능합니다.
- 그래서 알아야 합니다.

#######

# 2. 추가 판다스 기능들... 

## a. 파일을 읽어보아요. = 입출력 😏
---
- 일단 알고들 계시죠? pd.read_csv('주소값')
- csv 파일이 어떻게 생겼나요?

####### titanic.csv #######<br>
PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked<br>
1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.25,,S<br>
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38,1,0,PC 17599,71.2833,C85,C<br>
3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,STON/O2. 3101282,7.925,,S<br>
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35,1,0,113803,53.1,C123,S<br>
5,0,3,"Allen, Mr. William Henry",male,35,0,0,373450,8.05,,S<br>
6,0,3,"Moran, Mr. James",male,,0,0,330877,8.4583,,Q<br>
7,0,1,"McCarthy, Mr. Timothy J",male,54,0,0,17463,51.8625,E46,S<br>
#######################

In [59]:
df = pd.read_csv('https://raw.githubusercontent.com/tysep16/DV25_1/refs/heads/main/titanic.csv')
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


- 엑셀파일도 가능합니다. pd.read_excel('주소값')
- 단! openpyxl 패키지가 있어야 해요.<br><br>
<img src="https://raw.githubusercontent.com/tysep16/DV25_1/refs/heads/main/xlsx.jpg" width="600"><br>

In [60]:
!pip install openpyxl
df = pd.read_excel('https://github.com/tysep16/DV25_1/raw/refs/heads/main/titanic.xlsx')
df


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


,Unnamed: 0,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


- html의 테이블도 읽어올 수 있어요. --> 테이블 외 데이터는 beautifulsoup를 이용하세요.
- 단! lxml 패키지와 html5lib 패키지가 있어야 해요.
- ref: https://ko.wikipedia.org/wiki/%EB%8C%80%ED%95%9C%EB%AF%BC%EA%B5%AD%EC%9D%98_%EC%A0%80%EC%B6%9C%EC%82%B0

In [69]:
!pip install lxml
!pip install html5lib
df = pd.read_html('https://ko.wikipedia.org/wiki/%EB%8C%80%ED%95%9C%EB%AF%BC%EA%B5%AD%EC%9D%98_%EC%A0%80%EC%B6%9C%EC%82%B0')
df


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


[      연도    출산율
 0   2002  1.178
 1   2003  1.191
 2   2004  1.164
 3   2005  1.085
 4   2006  1.132
 5   2007  1.259
 6   2008  1.192
 7   2009  1.149
 8   2010  1.226
 9   2011  1.244
 10  2012  1.297
 11  2013  1.187
 12  2014  1.205
 13  2015  1.239
 14  2016  1.172
 15  2017  1.052
 16  2018  0.977
 17  2019  0.918
 18  2020  0.837
 19  2021  0.810,
           지역  출생아 수(천명)  조출생률  합계출산율  인구(2021년 기준)
 0         서울       47.4   5.0  0.642       9588711
 1         부산       15.1   4.5  0.747       3369704
 2         대구       11.2   4.6  0.807       2406296
 3         대전        7.5   5.1  0.829       1457619
 4         광주        7.3   5.1  0.811       1444787
 5         인천       16.0   5.5  0.829       2936214
 6        경기도       77.8   5.9  0.878      13479798
 7       전라북도        8.2   4.5  0.909       1796331
 8       경상남도       16.8   5.1  0.945       3329623
 9       충청북도        8.6   5.4  0.983       1596303
 10        울산        6.6   5.8  0.984       1128163
 11      경상북도     

- 테이블이 많아 리스트 형태로 출력 --> 우리가 필요한 테이블 찾기

In [62]:
df = df[2]
df

,지역/연도[6],2005,2006[7],2007,2008[8],2009[9],2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
0,서울,0.92,0.97,1.06,1.01,0.96,1.02,1.01,1.06,0.97,0.98,1.00,0.94,0.84,0.76,0.72,0.64,0.63
1,부산,0.88,0.91,1.02,0.98,0.94,1.05,1.08,1.14,1.05,1.09,1.14,1.10,0.98,0.90,0.83,0.75,0.73
2,대구,0.99,1.00,1.13,1.07,1.03,1.11,1.15,1.22,1.13,1.17,1.22,1.19,1.07,0.99,0.93,0.81,0.78
3,인천,1.07,1.11,1.25,1.19,1.14,1.21,1.23,1.30,1.20,1.21,1.22,1.14,1.01,1.01,0.94,0.83,0.78
4,광주,1.10,1.14,1.26,1.20,1.14,1.22,1.23,1.30,1.17,1.20,1.21,1.17,1.05,0.97,0.91,0.81,0.90
5,대전,1.10,1.15,1.27,1.22,1.16,1.21,1.26,1.32,1.23,1.25,1.28,1.19,1.08,0.95,0.88,0.81,0.81
6,울산,1.18,1.24,1.40,1.34,1.31,1.37,1.39,1.48,1.39,1.44,1.49,1.42,1.26,1.13,1.08,0.99,0.94
7,세종,-,-,-,-,-,-,-,1.60,1.44,1.35,1.89,1.82,1.67,1.57,1.47,1.28,1.28
8,경기,1.17,1.23,1.35,1.29,1.23,1.31,1.31,1.36,1.23,1.24,1.27,1.19,1.07,1.00,0.94,0.88,0.85
9,강원,1.18,1.19,1.35,1.25,1.25,1.31,1.34,1.37,1.25,1.25,1.31,1.24,1.12,1.07,1.08,1.04,0.98


## b. 데이터프레임을 저장해보아요. = 입출력 😏
---
- 일단 csv 파일로...

In [63]:
df.to_csv('df.csv')

- 엑셀 파일로...

In [64]:
df.to_excel('df.xlsx')

- 리스트는?

In [65]:
df.iloc[:10, :10].values.tolist()

[['서울', '0.92', '0.97', '1.06', '1.01', '0.96', '1.02', '1.01', 1.06, 0.97],
 ['부산', '0.88', '0.91', '1.02', '0.98', '0.94', '1.05', '1.08', 1.14, 1.05],
 ['대구', '0.99', '1.00', '1.13', '1.07', '1.03', '1.11', '1.15', 1.22, 1.13],
 ['인천', '1.07', '1.11', '1.25', '1.19', '1.14', '1.21', '1.23', 1.3, 1.2],
 ['광주', '1.10', '1.14', '1.26', '1.20', '1.14', '1.22', '1.23', 1.3, 1.17],
 ['대전', '1.10', '1.15', '1.27', '1.22', '1.16', '1.21', '1.26', 1.32, 1.23],
 ['울산', '1.18', '1.24', '1.40', '1.34', '1.31', '1.37', '1.39', 1.48, 1.39],
 ['세종', '-', '-', '-', '-', '-', '-', '-', 1.6, 1.44],
 ['경기', '1.17', '1.23', '1.35', '1.29', '1.23', '1.31', '1.31', 1.36, 1.23],
 ['강원', '1.18', '1.19', '1.35', '1.25', '1.25', '1.31', '1.34', 1.37, 1.25]]

## c. 마지막으로 짧게 describe() 🛣️
---
- 파일을 읽어오면: df를 보고 df.info()를 보고 그 다음이 df.describe() --> 컬럼방향

In [66]:
df.describe()

,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
count,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000
mean,1.398889,1.282222,1.294444,1.357778,1.291111,1.166667,1.082222,1.019444,0.928889,0.893889
std,0.163631,0.148544,0.140429,0.195946,0.192167,0.181853,0.173631,0.167454,0.152079,0.139627
min,1.060000,0.970000,0.980000,1.000000,0.940000,0.840000,0.760000,0.720000,0.640000,0.630000
25%,1.300000,1.192500,1.210000,1.225000,1.175000,1.055000,0.982500,0.922500,0.815000,0.810000
50%,1.405000,1.285000,1.290000,1.330000,1.245000,1.135000,1.055000,1.010000,0.930000,0.900000
75%,1.497500,1.387500,1.410000,1.475000,1.400000,1.260000,1.170000,1.087500,1.015000,0.957500
max,1.640000,1.520000,1.500000,1.890000,1.820000,1.670000,1.570000,1.470000,1.280000,1.280000


#######

# 3. 5주차 과제

## a. 다음 지시대로 빈 코드셀을 작성-->실행하여 캡처화면(jpg, png...)을 제출하세요. 😊
---
1. 다음 주소에서 타이타닉 데이터셋(허깅페이스 웹주)을 불러와서 데이터프레임 df를 만들어 보세요.
   - 주소: https://huggingface.co/datasets/BIT/titanic-dataset
   - 힌트: df = pd.?('?') + 불러온 테이블은 1개 뿐... 따라서 리스트 0번을 불러와야 함 + 100행만 불러와짐.

In [14]:
from datasets import load_dataset
import pandas as pd

# 허깅페이스에서 데이터셋 불러오기
dataset = load_dataset("BIT/titanic-dataset")

# 데이터셋은 딕셔너리 형태로, 'train' 키를 통해 접근
df = pd.DataFrame(dataset['train'][:100])  # 100개만 사용

2. 테이터프레임 df의 컬럼별 집계함수 결과를 출력하세요.
    - 힌트: df.?

In [15]:
# 집계 함수 적용
print(df.describe())

         survived      pclass        age       sibsp       parch        fare
count  100.000000  100.000000  78.000000  100.000000  100.000000  100.000000
mean     0.410000    2.400000  27.465769    0.730000    0.440000   29.517625
std      0.494311    0.816497  15.278878    1.179411    0.967346   40.972905
min      0.000000    1.000000   0.830000    0.000000    0.000000    7.225000
25%      0.000000    2.000000  18.250000    0.000000    0.000000    8.050000
50%      0.000000    3.000000  26.000000    0.000000    0.000000   15.675000
75%      1.000000    3.000000  34.750000    1.000000    0.000000   32.134375
max      1.000000    3.000000  71.000000    5.000000    5.000000  263.000000


3. df의 컬럼 이름들이 이상합니다. df.str을 이용하셔서 빈칸을 기준으로 split하고, 리스트의 첫번째 이름들을 컬럼 라벨로 새로 저장 & 출력하세요.
   - 힌트: (답 = 2줄)
   - df.columns = df.?.?.split(' ').?[?]
   - df.columns

In [16]:
# 공백 기준 split 후 첫 번째 요소만 추출
df.columns = df.columns.str.split(' ').str[0]
df.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone'],
      dtype='object')

4. df에서 1) 컬럼별로 결측치수를 출력해보고 2) age 컬럼의 결측치를 이전값으로 대체해 보세요.
   - 힌트: (답 = 2줄)
   - df.?.?
   - df.age.?(method=?)

In [17]:
# 1) 컬럼별 결측치 수 출력
print(df.isnull().sum())

survived        0
pclass          0
sex             0
age            22
sibsp           0
parch           0
fare            0
embarked        1
class           0
who             0
adult_male      0
deck           80
embark_town     1
alive           0
alone           0
dtype: int64


In [18]:
# 2) age 컬럼의 결측치를 앞의 값으로 대체
df['age'] = df['age'].fillna(method='ffill')

C:\Users\yunjh\AppData\Local\Temp\ipykernel_13280\53503430.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['age'] = df['age'].fillna(method='ffill')


5. df에서 살아남은 사람들 중(survived == 1) 성인 남자(adult_male)의 수를 groupby와 .agg(집계함수)를 써서 출력하세요.
   - 힌트: df.groupby(?)[?].agg([?])

In [19]:
# 조건에 맞는 데이터 필터링
survivors = df[df['survived'] == 1]

# groupby + agg 사용해서 성인 남성 수 계산
result = survivors.groupby('adult_male')['adult_male'].agg(['count'])
print(result)

            count
adult_male       
False          32
True            9
